In [ ]:
# Necessary packages
!pip install ucimlrepo
!pip install unidecode

In [25]:
import pandas as pd
import html
from ucimlrepo import fetch_ucirepo 

def repair_encoding(text):
    if not isinstance(text, str):
        return text

    cp1252_fix = {
        '\u20AC': '\x80', '\u201A': '\x82', '\u0192': '\x83', '\u201E': '\x84',
        '\u2026': '\x85', '\u2020': '\x86', '\u2021': '\x87', '\u02C6': '\x88',
        '\u2030': '\x89', '\u0160': '\x8A', '\u2039': '\x8B', '\u0152': '\x8C',
        '\u017D': '\x8E', '\u2018': '\x91', '\u2019': '\x92', '\u201C': '\x93',
        '\u201D': '\x94', '\u2022': '\x95', '\u2013': '\x96', '\u2014': '\x97',
        '\u02DC': '\x98', '\u2122': '\x99', '\u0161': '\x9A', '\u203A': '\x9B',
        '\u0153': '\x9C', '\u017E': '\x9E', '\u0178': '\x9F'
    }
    
    # 1. Replace Windows-1252 chars with their raw byte equivalents
    for char, byte_char in cp1252_fix.items():
        text = text.replace(char, byte_char)
        
    # 2. Encode to Latin-1 and decode as UTF-8
    try:
        return text.encode('latin-1').decode('utf-8')
        
    except UnicodeError:
        return text

# Fetch the data from the repository
data = fetch_ucirepo(id = 967)

# Complete the dataset for cleaning
raw_data = pd.concat([data.data.features, data.data.targets], axis = 1)

# Identify the columns which have str's
string_columns = raw_data.select_dtypes(include=['object']).columns

print(f"Checking columns: {list(string_columns)}\n")
print("-" * 40)

for col in string_columns:
    # Make a copy to compare
    original_series = raw_data[col].copy()
    
    # Apply the repair
    raw_data[col] = raw_data[col].apply(repair_encoding)
    
    # Compare to see what changed
    # (We compare null-safe by filling NAs with a placeholder)
    has_changed = (original_series.fillna('') != raw_data[col].fillna(''))
    num_changed = has_changed.sum()
    
    if num_changed > 0:
        print(f"MATCH FOUND in column '{col}': {num_changed} rows repaired.")
        # Show the first 2 changed examples
        changed_indices = raw_data[has_changed].index[:2]
        for idx in changed_indices:
            print(f"  Idx {idx} Before: {original_series[idx][:50]}...")
            print(f"  Idx {idx} After:  {raw_data[col][idx][:50]}...")
        print("-" * 40)
    else:
        print(f"Column '{col}': No repairs needed.")

Checking columns: ['URL', 'Domain', 'TLD', 'Title']

----------------------------------------
MATCH FOUND in column 'URL': 2 rows repaired.
  Idx 127839 Before: https://22017-5502.s3.webspace.re/privatkunden/Ã£Å...
  Idx 127839 After:  https://22017-5502.s3.webspace.re/privatkunden/ãœb...
  Idx 158380 Before: https://vinted.6545657.xyz/p5ci5kt9Ã¢â‚¬â€¹Ã¢â‚¬â€...
  Idx 158380 After:  https://vinted.6545657.xyz/p5ci5kt9â€‹â€‹â€‹â€‹â€‹...
----------------------------------------
Column 'Domain': No repairs needed.
Column 'TLD': No repairs needed.
MATCH FOUND in column 'Title': 8 rows repaired.
  Idx 0 Before: à¸‚à¹ˆà¸²à¸§à¸ªà¸” à¸‚à¹ˆà¸²à¸§à¸§à¸±à¸™à¸™à¸µà¹‰ ...
  Idx 0 After:  ข่าวสด ข่าววันนี้ ข่าวกีฬา ข่าวบันเทิง อัพเดทสดใหม...
  Idx 1 Before: johannes gutenberg-universitÃ¤t mainz...
  Idx 1 After:  johannes gutenberg-universität mainz...
----------------------------------------


In [28]:
import pandas as pd
import numpy as np

# Assuming 'raw_data' is your current dataframe

# 1. Length Consistency: Does URLLength match the actual length of the URL?
#    (We allow a small margin of error for potential whitespace trimming)
raw_data['Calculated_URLLength'] = (raw_data['URL'].str.len() - 1)
length_diff = raw_data[raw_data['URLLength'] != raw_data['Calculated_URLLength']]

# 2. Binary Logic: Is IsHTTPS actually consistent with the URL?
#    (Check if URL starts with https but IsHTTPS says 0, or vice versa)
https_mismatch = raw_data[
    (raw_data['URL'].str.startswith('https') & (raw_data['IsHTTPS'] == 0)) |
    (~raw_data['URL'].str.startswith('https') & (raw_data['IsHTTPS'] == 1))
]

# 3. Feature Logic: Does 'HasTitle' match whether the Title column is empty?
#    (Note: Pandas treats empty strings as distinct from NaN sometimes)
title_mismatch = raw_data[
    ((raw_data['Title'].isna() | (raw_data['Title'] == '')) & (raw_data['HasTitle'] == 1)) |
    ((raw_data['Title'].str.len() - 1 > 0) & (raw_data['HasTitle'] == 0))
]

# 4. Math Logic: Do ratios make sense? (e.g., LetterRatio = NoOfLetters / URLLength)
#    We check one example: LetterRatioInURL.
#    We round to 6 decimal places to avoid floating point errors.
raw_data['Calc_LetterRatio'] = raw_data['NoOfLettersInURL'] / raw_data['URLLength']
ratio_mismatch = raw_data[
    abs(raw_data['LetterRatioInURL'] - raw_data['Calc_LetterRatio']) > 0.0001
]

print(f"--- Consistency Report ---")
print(f"1. URL Length Mismatches: {len(length_diff)} rows")
print(f"2. HTTPS Mismatches:      {len(https_mismatch)} rows")
print(f"3. Title Flag Mismatches: {len(title_mismatch)} rows")
print(f"4. Letter Ratio Mismatches: {len(ratio_mismatch)} rows")

# Peek at the length mismatches if they exist (crucial for your encoding repair)
if len(length_diff) > 0:
    print("\nExample Length Mismatch (Repair likely changed string length):")
    print(length_diff[['URL', 'URLLength', 'Calculated_URLLength']].tail(10))

--- Consistency Report ---
1. URL Length Mismatches: 48646 rows
2. HTTPS Mismatches:      493 rows
3. Title Flag Mismatches: 0 rows
4. Letter Ratio Mismatches: 163352 rows

Example Length Mismatch (Repair likely changed string length):
                                                      URL  URLLength  \
235747  https://drivesafeamerica.us/wp-admin/dev/a32q0...         99   
235748  https://bamcobpl-pt.particul.pro/?utm_term=bpi...        123   
235752                               http://www.qcupn.com         20   
235759                             http://brigdedapps.net         22   
235760                               https://jpostcp.com/         20   
235766                     https://care56.weeblysite.com/         30   
235768  http://kratiknamdev.github.io/instagram-login-...         52   
235782  http://goldenrod-motley-texture.glitch.me/hvwa...         51   
235783        https://bancolombia.com1home0892.repl.co/?2         43   
235784                 https://aol-108318.we

In [57]:
# Cleaning data
processed_data = raw_data.drop_duplicates().dropna()


In [4]:
# Convert to csv and save
processed_data.to_csv('../data/processed/processed_dataset.csv', index=False)